##00-DEPENDÊNCIAS

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.functions import left

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

##01-Criação da Tabela BRONZE

In [0]:
volume_path = "/Volumes/data_warehouse/bndes/files_ingest"

files = dbutils.fs.ls(volume_path)
latest_file = sorted(files, key=lambda f: f.modificationTime)[-1]

print(f"Latest file name: {latest_file.name}")

if latest_file.name =="operacoes-financiamento-operacoes-nao-auto_carga.csv":
    print("Arquivo ja carregado")
    dbutils.jobs.taskValues.set(key = 'run_tsk', value = False)

elif "operacoes-financiamento-operacoes-nao-auto_20" in latest_file.name :
    dbutils.fs.cp(
    "/Volumes/data_warehouse/bndes/files_ingest/"+latest_file.name,
    "/Volumes/data_warehouse/bndes/files_ingest/operacoes-financiamento-operacoes-nao-auto_carga.csv")
    
    entry_file_raw = "/Volumes/data_warehouse/bndes/files_ingest/operacoes-financiamento-operacoes-nao-auto_carga.csv"
    file_data_ref = latest_file.name[-10:].replace(".csv", "")
    flag_run_task =  True
    
    print("Arquivo carregado:" +file_data_ref)

else :
    print("Arquivo não encontrado")
    dbutils.jobs.taskValues.set(key = 'run_tsk', value = False)

In [0]:
if flag_run_task:
    entry_file_schema = StructType([
        StructField("cliente", StringType(), True),
        StructField("cnpj", StringType(), True),
        StructField("descricao_do_projeto", StringType(), True),
        StructField("uf", StringType(), True),
        StructField("municipio", StringType(), True),
        StructField("municipio_codigo", StringType(), True),
        StructField("numero_do_contrato", StringType(), True),
        StructField("data_da_contratacao", StringType(), True),
        StructField("valor_contratado_reais", StringType(), True),
        StructField("valor_desenbolsado_reais", StringType(), True),
        StructField("origem_recurso_desembolsos", StringType(), True),
        StructField("custo_financeiro", StringType(), True),
        StructField("juros", StringType(), True),
        StructField("prazo_carencia_meses", StringType(), True),
        StructField("prazo_amortizacao_meses", StringType(), True),
        StructField("modalidade_apoio", StringType(), True),
        StructField("forma_de_apoio", StringType(), True),
        StructField("produto", StringType(), True),
        StructField("instrumento_financeiro", StringType(), True),
        StructField("inovacao", StringType(), True),
        StructField("area_operacional", StringType(), True),
        StructField("setor_cnae", StringType(), True),
        StructField("subsetor_cnae_agrupado", StringType(), True),
        StructField("subsetor_cnae_codigo", StringType(), True),
        StructField("subsetor_cnae_nome", StringType(), True),
        StructField("setor_bndes", StringType(), True),
        StructField("subsetor_bndes", StringType(), True),
        StructField("porte_do_cliente", StringType(), True),
        StructField("natureza_do_cliente", StringType(), True),
        StructField("instituicao_financeira_credenciada", StringType(), True),
        StructField("cnpj_instituicao_financeira_credenciada", StringType(), True),
        StructField("tipo_de_garantia", StringType(), True),
        StructField("tipo_de_excepcionalidade", StringType(), True),
        StructField("situacao_do_contrato", StringType(), True),
    ])

    entry_file = spark.read\
        .format("csv")\
        .option("header", "true")\
        .option("sep", ";")\
        .schema(entry_file_schema)\
        .option("encoding", "iso-8859-1")\
        .load(entry_file_raw)

    df_entry_file = entry_file\
        .withColumn("cnpj", 
            lpad(concat(substring(lpad(regexp_replace(col("cnpj"), "[/.,-]",""), 15, "0"),4,9),lit('***')),15,"*")
       )\
        .withColumn("numero_do_contrato", 
           lpad(regexp_replace(col("numero_do_contrato"), "[/.,-]",""), 9, "0")
        )\
       .withColumn("cnpj_instituicao_financeira_credenciada", 
           lpad(regexp_replace(col("cnpj_instituicao_financeira_credenciada"), "[/.,-]",""), 15, "0")
        )\
        .withColumn("data_da_contratacao", 
            date_format(to_date(col("data_da_contratacao"), "dd/MM/yyyy"), "yyyy-MM-dd")
        )\
        .withColumn("valor_contratado_reais", 
            regexp_replace(col("valor_contratado_reais"), ",",".").cast("double")
        )\
        .withColumn("valor_desenbolsado_reais", 
            regexp_replace(col("valor_desenbolsado_reais"), ",",".").cast("double")
        )\
        .withColumn("juros", 
            regexp_replace(col("juros"), ",",".").cast("double")
        )\
        .withColumn("prazo_carencia_meses", 
            regexp_replace(col("prazo_carencia_meses"), ",",".").cast("integer")
        )\
        .withColumn("prazo_amortizacao_meses", 
            regexp_replace(col("prazo_amortizacao_meses"), ",",".").cast("integer")
        )\
        .withColumn("cnpj_instituicao_financeira_credenciada", 
            lpad(concat(substring(lpad(regexp_replace(col("cnpj_instituicao_financeira_credenciada"), "[/.,-]",""), 15, "0"),4,8),lit('****')),15,"*")
        )\
        .withColumn("data_ref", 
            lit(file_data_ref)
        )

    dt_ref_bronze = spark.table("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]
    dt_ref_update = df_entry_file.agg(max(col("data_ref"))).first()[0]

    if dt_ref_update > dt_ref_bronze:
        try:
            print("appendando base atualizada")
            df_entry_file.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto")
        except:
            print("erro ao appendar")
    elif dt_ref_update == dt_ref_bronze:
        print("erro: base com referencia igual a ultima atualização")   
    else:
        print("erro: base com referencia anterior a ultima atualização")

else: print(flag_run_task)

##02-Criação da Tabela SILVER

In [0]:
if flag_run_task:
    dt_ref_bronze = spark.table("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]

    df_bronze_bndes = spark.table("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto").filter(col("data_ref").isin(dt_ref_bronze))

    df_silver_bndes = df_bronze_bndes.select(
        df_bronze_bndes.cnpj,
        df_bronze_bndes.cliente,
        df_bronze_bndes.descricao_do_projeto.alias("desc_proj"),
        df_bronze_bndes.uf.alias("sigla_uf"),
        df_bronze_bndes.municipio.alias("desc_municipio"),
        df_bronze_bndes.municipio_codigo.alias("cod_municipio"),
        df_bronze_bndes.numero_do_contrato.alias("cod_contrato"),
        df_bronze_bndes.data_da_contratacao.alias("dt_contrato"),
        df_bronze_bndes.valor_contratado_reais.alias("vlr_contrato"),
        df_bronze_bndes.valor_desenbolsado_reais.alias("vlr_desenbolso"),
        df_bronze_bndes.origem_recurso_desembolsos.alias("desc_orig_recurso"),
        df_bronze_bndes.custo_financeiro.alias("custo_finan"),
        concat(df_bronze_bndes.juros,lit("%")).alias("pct_juros"),
        df_bronze_bndes.prazo_carencia_meses.alias("prz_carencia_m"),
        df_bronze_bndes.prazo_amortizacao_meses.alias("prz_amort_m"),
        df_bronze_bndes.modalidade_apoio.alias("desc_modalidade_apoio"),
        df_bronze_bndes.forma_de_apoio.alias("desc_forma_apoio"),
        df_bronze_bndes.produto.alias("desc_produto"),
        df_bronze_bndes.instrumento_financeiro.alias("desc_instr_finan"),
        df_bronze_bndes.inovacao.alias("desc_inova"),
        df_bronze_bndes.area_operacional.alias("desc_area_op"),
        df_bronze_bndes.subsetor_cnae_codigo.alias("cod_cnae"),
        df_bronze_bndes.subsetor_cnae_nome.alias("desc_cnae"),
        df_bronze_bndes.setor_bndes.alias("desc_setor_bndes"),
        df_bronze_bndes.porte_do_cliente.alias("porte"),
        df_bronze_bndes.natureza_do_cliente.alias("desc_nat"),
        df_bronze_bndes.instituicao_financeira_credenciada.alias("if_nome"),
        df_bronze_bndes.cnpj_instituicao_financeira_credenciada.alias("if_cnpj"),
        df_bronze_bndes.tipo_de_garantia.alias("desc_garantia"),
        df_bronze_bndes.tipo_de_excepcionalidade.alias("desc_excep"),
        df_bronze_bndes.situacao_do_contrato.alias("situ_contrato"),
        df_bronze_bndes.data_ref
    )

    df_silver_bndes = df_silver_bndes.withColumn("if_nome", 
        when(df_silver_bndes.if_nome.isin("----------"), "BNDES"
        ).otherwise(df_silver_bndes.if_nome)
    )

    df_silver_bndes_grouped = df_silver_bndes.groupBy(
        df_silver_bndes.cnpj,
        df_silver_bndes.cliente,
        df_silver_bndes.cod_contrato,
        df_silver_bndes.dt_contrato,
        df_silver_bndes.desc_proj,
        df_silver_bndes.sigla_uf,
        df_silver_bndes.desc_municipio,
        df_silver_bndes.cod_municipio,
        df_silver_bndes.desc_orig_recurso,
        df_silver_bndes.custo_finan,
        df_silver_bndes.pct_juros,
        df_silver_bndes.prz_carencia_m,
        df_silver_bndes.prz_amort_m,
        df_silver_bndes.desc_modalidade_apoio,
        df_silver_bndes.desc_forma_apoio,
        df_silver_bndes.desc_produto,
        df_silver_bndes.desc_instr_finan,
        df_silver_bndes.desc_inova,
        df_silver_bndes.desc_area_op,
        df_silver_bndes.cod_cnae,
        df_silver_bndes.desc_cnae,
        df_silver_bndes.desc_setor_bndes,
        df_silver_bndes.porte,
        df_silver_bndes.desc_nat,
        df_silver_bndes.if_nome,
        df_silver_bndes.if_cnpj,
        df_silver_bndes.desc_garantia,
        df_silver_bndes.desc_excep,
        df_silver_bndes.situ_contrato,
        df_silver_bndes.data_ref
    ).agg(
        sum(df_silver_bndes.vlr_contrato).alias("vlr_contrato"),
        sum(df_silver_bndes.vlr_desenbolso).alias("vlr_desenbolso")
    )

    windowSpec = Window.partitionBy("cod_contrato").orderBy("cnpj")

    df_silver_bndes_sequ = df_silver_bndes_grouped\
        .withColumn("nr_seq", 
            row_number().over(windowSpec)
        )\
        .withColumn("cod_contrato_seq",
            concat_ws("_",col("cod_contrato"), col("nr_seq"))
        ).drop("nr_seq")

    df_silver_bndes_write = df_silver_bndes_sequ.select(
        df_silver_bndes_sequ.cnpj,
        df_silver_bndes_sequ.cliente,
        df_silver_bndes_sequ.desc_proj,
        df_silver_bndes_sequ.sigla_uf,
        df_silver_bndes_sequ.desc_municipio,
        df_silver_bndes_sequ.cod_municipio,
        df_silver_bndes_sequ.cod_contrato_seq,
        df_silver_bndes_sequ.dt_contrato,
        df_silver_bndes_sequ.vlr_contrato,
        df_silver_bndes_sequ.vlr_desenbolso,
        df_silver_bndes_sequ.desc_orig_recurso,
        df_silver_bndes_sequ.custo_finan,
        df_silver_bndes_sequ.pct_juros,
        df_silver_bndes_sequ.prz_carencia_m,
        df_silver_bndes_sequ.prz_amort_m,
        df_silver_bndes_sequ.desc_modalidade_apoio,
        df_silver_bndes_sequ.desc_forma_apoio,
        df_silver_bndes_sequ.desc_produto,
        df_silver_bndes_sequ.desc_instr_finan,
        df_silver_bndes_sequ.desc_inova,
        df_silver_bndes_sequ.desc_area_op,
        df_silver_bndes_sequ.cod_cnae,
        df_silver_bndes_sequ.desc_cnae,
        df_silver_bndes_sequ.desc_setor_bndes,
        df_silver_bndes_sequ.porte,
        df_silver_bndes_sequ.desc_nat,
        df_silver_bndes_sequ.if_nome,
        df_silver_bndes_sequ.if_cnpj,
        df_silver_bndes_sequ.desc_garantia,
        df_silver_bndes_sequ.desc_excep,
        df_silver_bndes_sequ.situ_contrato,
        df_silver_bndes_sequ.data_ref
    )

    dt_ref_silver = spark.table("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]
    dt_ref_update = df_silver_bndes_write.agg(max(col("data_ref"))).first()[0]

    if dt_ref_update > dt_ref_silver:
        try:
            print("appendando base atualizada")
            df_silver_bndes_write.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto")
        except:
            print("erro ao appendar")
    elif dt_ref_update == dt_ref_silver:
        print("erro: base com referencia igual a ultima atualização")
    else:
        print("erro: base com referencia anterior a ultima atualização")

else: print(flag_run_task)

#03-Criação da Tabela Gold

In [0]:
if flag_run_task:
    dt_ref_silver = spark.table("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]

    df_silver_bndes = spark.table("data_warehouse.bndes.SILVER_LAYER_bndes_oper_finan_n_auto").filter(col("data_ref").isin(dt_ref_silver))

    df_gold = df_silver_bndes

    df_gold_bndes_write = df_gold.groupBy(
        df_gold.data_ref,
        substring(df_gold.dt_contrato,1,7).alias("anomes_contrato"),
        df_gold.sigla_uf,
        df_gold.desc_municipio,
        df_gold.desc_orig_recurso,
        df_gold.situ_contrato,
        df_gold.desc_garantia,
        df_gold.if_nome,
        df_gold.porte,
        df_gold.desc_setor_bndes,
        df_gold.desc_cnae,
        df_gold.desc_area_op,
        df_gold.desc_inova,
        df_gold.desc_produto,
        df_gold.desc_forma_apoio,
        df_gold.desc_modalidade_apoio
    ).agg(
        count(df_gold.cod_contrato_seq).alias("qt_contrato_seq"),
        avg(regexp_replace(col("pct_juros"), "%","").cast("float")).alias("avg_pct_juros"),
        sum(df_gold.vlr_contrato).alias("sum_vlr_contrato")
    )

    dt_ref_gold = spark.table("data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]
    dt_ref_update = df_gold_bndes_write.agg(max(col("data_ref"))).first()[0]

    if dt_ref_update > dt_ref_gold:
        try:
            print("appendando base atualizada")
            df_gold_bndes_write.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto")
        except:
            print("erro ao appendar")
    elif dt_ref_update == dt_ref_gold:
        print("erro: base com referencia igual a ultima atualização")
    else:
        print("erro: base com referencia anterior a ultima atualização")
else: print(flag_run_task)

In [0]:
spark.sql("""
    ALTER TABLE data_warehouse.bndes.GOLD_LAYER_bndes_oper_finan_n_auto
    DROP PARTITION (data_ref = '202602')
""")